# Machine Learning Project
# Kylle Waldie

# Pokemon Grading Tool

## Web Scraping

### Config

In [14]:
import requests
from bs4 import BeautifulSoup
import time
import re
import csv
import os

# HTTP headers to mimic a real browser
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9"
}

# eBay search URL for sold Pokémon PSA cards
BASE_SEARCH_URL = (
    "https://www.ebay.com/sch/i.html"
    "?_nkw=PSA+Pokemon+card"
    "&LH_Sold=1"
    "&LH_Complete=1"
)

# Where to save images and CSV
IMAGE_DIR = "data/images"
CSV_FILE = "data/labels.csv"

# Make sure directories exist
os.makedirs(IMAGE_DIR, exist_ok=True)
os.makedirs("data", exist_ok=True)

### Helper Functions

In [15]:
def get_soup(url):
    """Get BeautifulSoup object from a URL"""
    r = requests.get(url, headers=HEADERS, timeout=10)
    r.raise_for_status()
    return BeautifulSoup(r.text, "html.parser")

def extract_grade(title):
    """Extract PSA grade from listing title"""
    match = re.search(r"PSA\s?(\d+)", title)
    return match.group(1) if match else None

def upgrade_image_url(url):
    """Convert thumbnail to high-res eBay image"""
    return re.sub(r"s-l\d+", "s-l1600", url)

def download_image(url, filename):
    """Download image from URL to local file"""
    r = requests.get(url, headers=HEADERS, stream=True)
    if r.status_code == 200:
        with open(filename, "wb") as f:
            for chunk in r.iter_content(1024):
                f.write(chunk)

### Scrape Logic

In [31]:
def scrape():
    with open(CSV_FILE, "w", newline="", encoding="utf-8") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["filename", "grade", "listing_url"])

        # Get the search results page
        soup = get_soup(BASE_SEARCH_URL)

        # Find all listing links
        links = soup.select('a[href*="/itm/"]')

        # Clean URLs and deduplicate
        listing_urls = list(set(
            link.get("href").split("?")[0]
            for link in links
            if link.get("href")
        ))

        print(f"Found {len(listing_urls)} listing URLs")
        
        # Loop over each listing URL
        for idx, listing_url in enumerate(listing_urls):
            print(f"Processing {idx+1}/{len(listing_urls)}: {listing_url}")
            
            try:
                listing_soup = get_soup(listing_url)

                # Get the title and grade
                title_tag = listing_soup.select_one("#itemTitle")
                if not title_tag:
                    continue

                title = title_tag.get_text(strip=True).replace("Details about", "")
                grade = extract_grade(title)
                if not grade:
                    continue

                # Get the main image
                img_tag = listing_soup.select_one("#icImg")
                if not img_tag:
                    continue

                img_url = upgrade_image_url(img_tag.get("src"))
                filename = f"psa_{grade}_{idx}.jpg"
                filepath = os.path.join(IMAGE_DIR, filename)

                download_image(img_url, filepath)

                # Save metadata
                writer.writerow([filename, grade, listing_url])
                print(f"Downloaded PSA {grade} → {filename}")

                # Pause to avoid being blocked
                time.sleep(2)

            except Exception as e:
                print(f"Skipping listing due to error: {e}")
                continue

IndentationError: unindent does not match any outer indentation level (<string>, line 18)

### Running Scrape()

In [23]:
scrape()

Found 60 listing URLs


NameError: name 'link' is not defined